# October Task #5: Support Table and Class Weights

**What this produces**
1. **Support table:** for every category and split (train, validation): unmasked positives and negatives, masked pairs, and the number of **distinct context groups** behind each.
2. **Both-classes check:** every category must have positives *and* negatives (G2: "Confirm that each category has both classes").
3. **Class weights:** `pos_weight = negatives / positives` per category, computed from the **training split only** and saved in the fixed category order (the formula frozen in Task 1).

**Input:** `task3_chunk_category_targets.csv` from the Task 3 notebook (train + validation rows; the official test set is never part of it).

**Rules this notebook follows**
- Masked pairs are excluded from every count and from the weights.
- Validation rows never enter the weights. The official test set is never read.
- Nothing is written to the repo. Outputs go to `october_task5_outputs/`; the last cell zips them for download.

**Runtime:** CPU. Run all cells top to bottom.

## 1. Setup and load the Task 3 table

In [3]:
import hashlib
import json
from pathlib import Path

import numpy as np
import pandas as pd

BASE_URL = (
    "https://raw.githubusercontent.com/"
    "Break-Through-Tech/Accenture-1O-contract-review-challenge/main/"
)
OUTPUT_DIR = Path("october_task5_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

# Fixed category order (same as Task 1 frozen spec / Task 10 / Amie's FINAL_IDS)
CATS = [
    "governing_law", "renewal_term", "revenue_profit_sharing", "cap_on_liability",
    "uncapped_liability", "termination_for_convenience", "anti_assignment",
    "audit_rights", "license_grant", "exclusivity",
]

checks = []

def add_check(section, check, expected, actual, passed=None, note=""):
    if passed is None:
        passed = (actual == expected)
    checks.append({"section": section, "check": check, "expected": expected,
                   "actual": actual, "passed": bool(passed), "note": note})

def show_checks(section):
    df = pd.DataFrame(checks)
    df = df[df["section"] == section].reset_index(drop=True)
    display(df)
    n_fail = int((~df["passed"]).sum())
    print(f"{section}: {len(df) - n_fail}/{len(df)} checks passed" + ("" if n_fail == 0 else f"  <-- {n_fail} FAILED, investigate"))

TARGETS_PATH = Path("october_task3_outputs/task3_chunk_category_targets.csv")
if not TARGETS_PATH.exists():
    try:
        from google.colab import files
        print("Upload task3_chunk_category_targets.csv (from the Task 3 outputs zip).")
        uploaded = files.upload()
        TARGETS_PATH.parent.mkdir(exist_ok=True)
        Path(next(iter(uploaded))).replace(TARGETS_PATH)
    except ImportError:
        raise FileNotFoundError(f"Put the Task 3 targets CSV at {TARGETS_PATH}")

targets_sha256 = hashlib.sha256(TARGETS_PATH.read_bytes()).hexdigest()
df = pd.read_csv(TARGETS_PATH)
print("rows:", len(df), "| columns:", df.columns.tolist())
print("targets file sha256:", targets_sha256)
print(df["split"].value_counts().to_string())

Upload task3_chunk_category_targets.csv (from the Task 3 outputs zip).


Saving task3_chunk_category_targets.csv to task3_chunk_category_targets.csv
rows: 137140 | columns: ['split', 'chunk_id', 'contract_id', 'context_group_id', 'chunk_number', 'category_id', 'target', 'n_contained_spans', 'n_partial_spans', 'label', 'is_masked', 'annotation_present']
targets file sha256: 8fdae637a8940797c21c500e5bcb06d83f57be1a0d02c905127ff6cd15ac6833
split
train         108410
validation     28730


## 2. Validate the input table

In [5]:
S = "2_input"
required = {"split", "chunk_id", "contract_id", "context_group_id", "category_id", "target"}
add_check(S, "required_columns_present", [], sorted(required - set(df.columns)))
add_check(S, "splits_are_exactly_train_and_validation", ["train", "validation"], sorted(df["split"].unique()),
          note="the official test set must not be in this table")
add_check(S, "categories_are_the_10_selected", sorted(CATS), sorted(df["category_id"].unique()))
add_check(S, "targets_only_positive_negative_masked", [], sorted(set(df["target"].unique()) - {"positive", "negative", "masked"}))
add_check(S, "duplicate_chunk_category_rows", 0, int(df.duplicated(["chunk_id", "category_id"]).sum()))
rows_per_chunk = df.groupby("chunk_id")["category_id"].nunique()
add_check(S, "every_chunk_has_exactly_10_category_rows", True, bool((rows_per_chunk == 10).all()),
          note=f"{len(rows_per_chunk)} chunks")

tr, va = df[df["split"] == "train"], df[df["split"] == "validation"]
add_check(S, "no_chunk_in_both_splits", 0, len(set(tr["chunk_id"]) & set(va["chunk_id"])))
add_check(S, "no_contract_in_both_splits", 0, len(set(tr["contract_id"]) & set(va["contract_id"])))
add_check(S, "no_context_group_in_both_splits", 0, len(set(tr["context_group_id"]) & set(va["context_group_id"])))
add_check(S, "validation_context_groups", 83, int(va["context_group_id"].nunique()), note="frozen split: 83 validation groups")
show_checks(S)

,section,check,expected,actual,passed,note
0,2_input,required_columns_present,[],[],True,
1,2_input,splits_are_exactly_train_and_validation,"[train, validation]","[train, validation]",True,the official test set must not be in this table
2,2_input,categories_are_the_10_selected,"[anti_assignment, audit_rights, cap_on_liabili...","[anti_assignment, audit_rights, cap_on_liabili...",True,
3,2_input,targets_only_positive_negative_masked,[],[],True,
4,2_input,duplicate_chunk_category_rows,0,0,True,
5,2_input,every_chunk_has_exactly_10_category_rows,True,True,True,13714 chunks
6,2_input,no_chunk_in_both_splits,0,0,True,
7,2_input,no_contract_in_both_splits,0,0,True,
8,2_input,no_context_group_in_both_splits,0,0,True,
9,2_input,validation_context_groups,83,83,True,frozen split: 83 validation groups


2_input: 10/10 checks passed


## 3. Support table

For each split and category (unmasked pairs only count toward positives and negatives):

| column | meaning |
|---|---|
| `positive`, `negative` | unmasked chunk-category pairs with that target |
| `masked` | pairs excluded everywhere (boundary-split or no usable offsets) |
| `groups_with_positive` | distinct `context_group_id` with at least one positive chunk |
| `groups_with_negative` | distinct `context_group_id` with at least one unmasked negative chunk |
| `groups_total_in_split` | distinct groups in the split |

In [6]:
def support_table(frame):
    out = []
    for split in ["train", "validation"]:
        d = frame[frame["split"] == split]
        for cat in CATS:
            c = d[d["category_id"] == cat]
            pos = c[c["target"] == "positive"]
            neg = c[c["target"] == "negative"]
            msk = c[c["target"] == "masked"]
            n_unmasked = len(pos) + len(neg)
            out.append({
                "split": split, "category_id": cat,
                "positive": len(pos), "negative": len(neg), "masked": len(msk),
                "unmasked_total": n_unmasked,
                "positive_rate_pct": round(100 * len(pos) / n_unmasked, 3) if n_unmasked else np.nan,
                "groups_with_positive": int(pos["context_group_id"].nunique()),
                "groups_with_negative": int(neg["context_group_id"].nunique()),
                "groups_total_in_split": int(d["context_group_id"].nunique()),
                "contracts_with_positive": int(pos["contract_id"].nunique()),
            })
    return pd.DataFrame(out)

support = support_table(df)
support["thin_support_info"] = support["positive"] < 10     # informational flag only
pd.set_option("display.width", 220)
display(support)

,split,category_id,positive,negative,masked,unmasked_total,positive_rate_pct,groups_with_positive,groups_with_negative,groups_total_in_split,contracts_with_positive,thin_support_info
0,train,governing_law,291,10539,11,10830,2.687,272,323,324,273,False
1,train,renewal_term,127,10709,5,10836,1.172,119,324,324,119,False
2,train,revenue_profit_sharing,178,10628,35,10806,1.647,98,324,324,98,False
3,train,cap_on_liability,333,10456,52,10789,3.086,179,324,324,179,False
4,train,uncapped_liability,91,10732,18,10823,0.841,73,324,324,73,False
5,train,termination_for_convenience,143,10690,8,10833,1.320,121,324,324,122,False
6,train,anti_assignment,313,10509,19,10822,2.892,234,324,324,234,False
7,train,audit_rights,259,10569,13,10828,2.392,139,324,324,139,False
8,train,license_grant,386,10417,38,10803,3.573,162,323,324,162,False
9,train,exclusivity,203,10620,18,10823,1.876,117,324,324,117,False


## 4. Both classes present

Each category must have at least one unmasked positive **and** one unmasked negative, with at least one supporting group on each side. If this fails in the **train** split, stop: a weight cannot be computed and the category needs investigation. A **validation** failure means that category's metrics and threshold cannot be estimated.

In [7]:
S = "4_both_classes"
for split in ["train", "validation"]:
    s = support[support["split"] == split]
    missing = s[(s["positive"] == 0) | (s["negative"] == 0) | (s["groups_with_positive"] == 0) | (s["groups_with_negative"] == 0)]["category_id"].tolist()
    add_check(S, f"{split}_categories_missing_a_class", [], missing)
thin = support[(support["split"] == "validation") & (support["positive"] < 10)][["category_id", "positive", "groups_with_positive"]]
if len(thin):
    print("INFO: validation categories with fewer than 10 positive chunks (metrics will be noisy):")
    display(thin)
show_checks(S)
assert all(c["passed"] for c in checks if c["section"] == S and c["check"].startswith("train")), \
    "A training category lacks positives or negatives: stop and investigate before computing weights."

,section,check,expected,actual,passed,note
0,4_both_classes,train_categories_missing_a_class,[],[],True,
1,4_both_classes,validation_categories_missing_a_class,[],[],True,


4_both_classes: 2/2 checks passed


## 5. Class weights (training data only)

`pos_weight[c] = (# unmasked negative train pairs for c) / (# unmasked positive train pairs for c)`

This is the formula frozen in Task 1 (`positive_weight_formula`, `weight_data_source: Training data only`), and the form PyTorch `BCEWithLogitsLoss(pos_weight=...)` expects: it multiplies the loss on positive labels. Example: 900 negatives and 100 positives gives 9.

The function refuses any input that contains a split other than `train`.

In [8]:
def compute_pos_weight(train_frame):
    # Training-only class weights, in CATS order. Masked pairs are excluded.
    assert set(train_frame["split"].unique()) == {"train"}, \
        f"weights must be computed from the train split only, got {sorted(train_frame['split'].unique())}"
    weights, counts = {}, {}
    for cat in CATS:
        c = train_frame[train_frame["category_id"] == cat]
        p = int((c["target"] == "positive").sum())
        n = int((c["target"] == "negative").sum())
        if p == 0 or n == 0:
            raise ValueError(f"{cat}: {p} positives, {n} negatives. Stop and investigate; do not substitute a weight.")
        weights[cat] = n / p
        counts[cat] = {"positive": p, "negative": n}
    return weights, counts

pos_weight, weight_counts = compute_pos_weight(df[df["split"] == "train"])
weights_table = pd.DataFrame({
    "category_id": CATS,
    "train_positive": [weight_counts[c]["positive"] for c in CATS],
    "train_negative": [weight_counts[c]["negative"] for c in CATS],
    "pos_weight": [pos_weight[c] for c in CATS],
})
weights_table["pos_weight"] = weights_table["pos_weight"].round(4)
display(weights_table)

,category_id,train_positive,train_negative,pos_weight
0,governing_law,291,10539,36.2165
1,renewal_term,127,10709,84.3228
2,revenue_profit_sharing,178,10628,59.7079
3,cap_on_liability,333,10456,31.3994
4,uncapped_liability,91,10732,117.9341
5,termination_for_convenience,143,10690,74.7552
6,anti_assignment,313,10509,33.5751
7,audit_rights,259,10569,40.8069
8,license_grant,386,10417,26.9870
9,exclusivity,203,10620,52.3153


### Why this treatment does not duplicate validation or official-test examples

- `pos_weight` is a **single number per category that scales the loss on positive labels**. It does not oversample, resample, copy or augment any row, so no example appears more than once in training because of it.
- The counts and weights come only from rows where `split == "train"` (enforced by an assertion in `compute_pos_weight`). Validation rows are used for model and threshold selection only, and official-test rows are never read here.
- No contract or `context_group_id` appears in both splits (checked in Section 2), so train-derived weights carry no information from validation or test contracts.
- The weights are saved with the SHA-256 of the Task 3 table they came from, so reviewers can verify they were not recomputed from different data.

## 6. Validate the weights

In [9]:
S = "6_validate_weights"

# independent recomputation with a pivot (different code path)
piv = (df[(df["split"] == "train") & (df["target"] != "masked")]
       .pivot_table(index="category_id", columns="target", values="chunk_id", aggfunc="count", fill_value=0)
       .reindex(CATS))
indep = (piv["negative"] / piv["positive"]).to_numpy()
mine = np.array([pos_weight[c] for c in CATS])
add_check(S, "weights_match_independent_pivot_recompute", True, bool(np.allclose(mine, indep, rtol=0, atol=1e-12)))
add_check(S, "ten_weights_in_category_order", 10, len(mine))
add_check(S, "all_weights_finite_and_positive", True, bool(np.isfinite(mine).all() and (mine > 0).all()))
add_check(S, "weights_computed_from_train_rows_only", {"train"}, set(df[df["split"] == "train"]["split"].unique()))
add_check(S, "masked_pairs_excluded_from_counts", True,
          bool(all(weight_counts[c]["positive"] + weight_counts[c]["negative"] ==
                   int(((df["split"] == "train") & (df["category_id"] == c) & (df["target"] != "masked")).sum()) for c in CATS)))

# regression against the committed Task 10 train summary
try:
    ref = pd.read_csv(BASE_URL + "notebooks/task10_outputs/target_summary_by_category.csv").set_index("category_id")
    tr_sum = (df[df["split"] == "train"].pivot_table(index="category_id", columns="target", values="chunk_id", aggfunc="count", fill_value=0)
              .reindex(CATS).reindex(columns=["positive", "masked", "negative"], fill_value=0))
    same = bool((tr_sum.loc[ref.index, ["positive", "masked", "negative"]].to_numpy()
                 == ref[["positive", "masked", "negative"]].to_numpy()).all())
    add_check(S, "train_counts_equal_task10_summary", True, same, note="notebooks/task10_outputs/target_summary_by_category.csv")
    ref_w = (ref["negative"] / ref["positive"]).reindex(CATS).to_numpy()
    add_check(S, "weights_equal_task10_negative_over_positive", True, bool(np.allclose(mine, ref_w)))
except Exception as exc:
    add_check(S, "task10_reference_loaded", True, False, note=repr(exc))
show_checks(S)

,section,check,expected,actual,passed,note
0,6_validate_weights,weights_match_independent_pivot_recompute,True,True,True,
1,6_validate_weights,ten_weights_in_category_order,10,10,True,
2,6_validate_weights,all_weights_finite_and_positive,True,True,True,
3,6_validate_weights,weights_computed_from_train_rows_only,{train},{train},True,
4,6_validate_weights,masked_pairs_excluded_from_counts,True,True,True,
5,6_validate_weights,train_counts_equal_task10_summary,True,True,True,notebooks/task10_outputs/target_summary_by_cat...
6,6_validate_weights,weights_equal_task10_negative_over_positive,True,True,True,


6_validate_weights: 7/7 checks passed


## 7. Save outputs

In [10]:
support.to_csv(OUTPUT_DIR / "task5_support_table.csv", index=False)
weights_table.to_csv(OUTPUT_DIR / "task5_class_weights.csv", index=False)

weights_json = {
    "task": "October Task 5: support table and class weights",
    "formula": "pos_weight = number of unmasked negative training pairs / number of unmasked positive training pairs",
    "unit": "chunk-category pair",
    "source_split": "train",
    "masked_pairs_excluded": True,
    "imbalance_treatment": "positive-class weighting of BCEWithLogitsLoss; no oversampling, resampling or duplication of rows",
    "category_order": CATS,
    "pos_weight": {c: pos_weight[c] for c in CATS},
    "pos_weight_list": [pos_weight[c] for c in CATS],
    "train_counts": weight_counts,
    "n_train_chunks": int(df[df["split"] == "train"]["chunk_id"].nunique()),
    "targets_file": str(TARGETS_PATH),
    "targets_file_sha256": targets_sha256,
}
(OUTPUT_DIR / "task5_class_weights.json").write_text(json.dumps(weights_json, indent=2))
pd.DataFrame(checks).to_csv(OUTPUT_DIR / "task5_validation_checks.csv", index=False)

failed = [c for c in checks if not c["passed"]]
print(f"{len(checks) - len(failed)}/{len(checks)} checks passed")
for c in failed:
    print("FAILED:", c["section"], c["check"], "| expected:", c["expected"], "| actual:", c["actual"], "|", c["note"])
print("\nFiles written:")
for p in sorted(OUTPUT_DIR.iterdir()):
    print(" ", p, f"({p.stat().st_size / 1e6:.2f} MB)")

19/19 checks passed

Files written:
  october_task5_outputs/task5_class_weights.csv (0.00 MB)
  october_task5_outputs/task5_class_weights.json (0.00 MB)
  october_task5_outputs/task5_support_table.csv (0.00 MB)
  october_task5_outputs/task5_validation_checks.csv (0.00 MB)


In [11]:
import shutil
shutil.make_archive("october_task5_outputs", "zip", OUTPUT_DIR)
try:
    from google.colab import files
    files.download("october_task5_outputs.zip")
except Exception as exc:
    print("Not running in Colab; zip is at october_task5_outputs.zip", repr(exc))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Task 6 (For Amie)





Replace the placeholder weights in the harness with:

```python
import json, torch

w = json.load(open("october_task5_outputs/task5_class_weights.json"))
assert w["category_order"] == FINAL_IDS            # same order as the label and mask columns
pos_weight = torch.tensor(w["pos_weight_list"], dtype=torch.float32)   # shape (10,)
```